In [ ]:
# Cell: setup (run once)
!pip install --extra-index-url=https://pypi.nvidia.com cudf-polars-cu12

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %load_ext cudf.pandas
import pandas as pd
import cudf
import polars as pl
print("cuDF Pandas GPU mode is active!")

/usr/local/lib/python3.13/dist-packages/cudf/pandas/__init__.py:59: UserWarning: Failed to check cudaDevAttrConcurrentManagedAccess with error <cudaError_t.cudaErrorInsufficientDriver: 35>
  warnings.warn(str(e))


cuDF Pandas GPU mode is active!


Part A

In [ ]:
df_lazy = pl.scan_parquet(
    "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/HI-Large_Cleaned.parquet"
)

q_transformed = df_lazy.with_columns([
    pl.col("Timestamp").dt.hour().alias("Transaction_Hour"),
    pl.col("Timestamp").dt.weekday().alias("Day_Of_Week"),
    pl.col("Timestamp").dt.month().alias("Month"),
    pl.col("Timestamp").dt.weekday().is_in([4, 5]).cast(pl.Int32).alias("Is_Weekend"),

    pl.col("Amount Paid").log1p().alias("Log_Amount_Paid"),
    pl.col("Amount Received").log1p().alias("Log_Amount_Received"),
    (pl.col("Amount Paid") - pl.col("Amount Received")).alias("Amount_Difference"),
    (pl.col("Amount Paid") / (pl.col("Amount Received") + 1)).alias("Amount_Ratio"),
])

output_path = "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/q_transformed.parquet"

q_transformed.sink_parquet(output_path, compression="zstd")

print(f"✅ Section B: q_transformed features successfully generated and saved to:\n{output_path}")

✅ Section B: q_transformed features successfully generated and saved to:
/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/q_transformed.parquet


In [ ]:
df_lazy = pl.scan_parquet(
    "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/HI-Large_Cleaned.parquet"
)
df_lazy.select(pl.len()).collect()

len
u32
181294816


In [ ]:
df_lazy = pl.scan_parquet(
    "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/q_transformed.parquet"
)
df_lazy.select(pl.len()).collect()

len
u32
181294816


Part B

In [ ]:

import polars as pl
from datetime import timedelta

In [ ]:


df_lazy = pl.scan_parquet(
    "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/HI-Large_Cleaned.parquet"
)

df_lazy = df_lazy.with_columns([
    pl.col("Timestamp").dt.date().alias("Transaction_Date"),
    pl.col("Timestamp").dt.truncate("1w").alias("Week"),
])

min_date = (
    df_lazy.select(pl.col("Transaction_Date").min())
    .collect(engine="gpu")
    .item()
)
cutoff_date = min_date + timedelta(days=90)

print(f"Testing window: {min_date} → {cutoff_date}")

df_lazy = df_lazy.filter(pl.col("Transaction_Date") < cutoff_date)

Testing window: 2022-08-01 → 2022-10-30


In [ ]:
sender_features_lazy = (
    df_lazy.group_by(["Account", "Week"])
    .agg(
        pl.len().alias("outgoing_transaction_count"),
        pl.col("Amount Paid").sum().alias("total_amount_sent"),
        pl.col("Amount Paid").mean().alias("avg_amount_sent"),
        pl.col("Amount Paid").max().alias("max_amount_sent"),
        pl.col("Account_duplicated_0").n_unique().alias("unique_receivers"),
        (pl.col("Transaction_Date").max() - pl.col("Transaction_Date").min())
            .dt.total_days().alias("outgoing_active_days")
    )
    .with_columns(
        (pl.col("outgoing_transaction_count") / 7.0)
        .alias("outgoing_transaction_frequency")
    )
    .select([
        "Account", "Week",
        "outgoing_transaction_count", "total_amount_sent", "avg_amount_sent",
        "max_amount_sent", "unique_receivers", "outgoing_transaction_frequency"
    ])
)

sender_features = sender_features_lazy.collect(engine="gpu")
sender_features.write_parquet("/content/sender_features_weekly_test.parquet", compression="zstd")
print(f"Sender rows: {sender_features.height}")

Sender rows: 15878267


In [ ]:
receiver_features_lazy = (
    df_lazy.group_by(["Account_duplicated_0", "Week"])
    .agg(
        pl.len().alias("incoming_transaction_count"),
        pl.col("Amount Received").sum().alias("total_amount_received"),
        pl.col("Amount Received").mean().alias("avg_amount_received"),
        pl.col("Amount Received").max().alias("max_amount_received"),
        pl.col("Account").n_unique().alias("unique_senders"),
        (pl.col("Transaction_Date").max() - pl.col("Transaction_Date").min())
            .dt.total_days().alias("incoming_active_days")
    )
    .with_columns(
        (pl.col("incoming_transaction_count") / 7.0)
        .alias("incoming_transaction_frequency")
    )
    .select([
        "Account_duplicated_0", "Week",
        "incoming_transaction_count", "total_amount_received", "avg_amount_received",
        "max_amount_received", "unique_senders", "incoming_transaction_frequency"
    ])
)

receiver_features = receiver_features_lazy.collect(engine="gpu")
receiver_features.write_parquet("/content/receiver_features_weekly_test.parquet", compression="zstd")
print(f"Receiver rows: {receiver_features.height}")

Receiver rows: 14884322


In [ ]:
sender = pl.read_parquet("/content/sender_features_weekly_test.parquet")
receiver = pl.read_parquet("/content/receiver_features_weekly_test.parquet")

behavioral_features = (
    sender.lazy()
    .join(
        receiver.lazy(),
        left_on=["Account", "Week"],
        right_on=["Account_duplicated_0", "Week"],
        how="full",
    )
    .with_columns(pl.coalesce(["Account", "Account_duplicated_0"]).alias("Final_Account"))
    .drop(["Account", "Account_duplicated_0"])
    .fill_null(0)
    .collect(engine="gpu")
)

output_path = "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/Behavioral_Features_Account_Level_Weekly_TEST.parquet"
behavioral_features.write_parquet(output_path, compression="zstd")
print(f"✅ Test weekly Behavioral Features saved to:\n{output_path}")
print(f"Total rows: {behavioral_features.height}")

✅ Test weekly Behavioral Features saved to:
/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/Behavioral_Features_Account_Level_Weekly_TEST.parquet
Total rows: 21297791


In [ ]:
upsample_dense = True

if upsample_dense:
    all_weeks = behavioral_features.select("Week").unique().sort("Week")
    all_accounts = behavioral_features.select("Final_Account").unique()
    dense_grid = all_accounts.join(all_weeks, how="cross")

    behavioral_features = (
        dense_grid.lazy()
        .join(behavioral_features.lazy(), on=["Final_Account", "Week"], how="left")
        .fill_null(0)
        .collect(engine="gpu")
    )

ROLL_WINDOW = 4  # trailing 4 weeks

amount_cols = [
    "max_amount_sent", "max_amount_received",
    "total_amount_sent", "total_amount_received",
    "avg_amount_sent", "avg_amount_received",
]

behavioral_features = behavioral_features.sort(["Final_Account", "Week"])

lag_exprs = []
roll_mean_exprs = []
roll_std_exprs = []

for col in amount_cols:
    lag_exprs.append(
        pl.col(col).shift(1).over("Final_Account").alias(f"{col}_prev_week")
    )
    roll_mean_exprs.append(
        pl.col(col).shift(1).rolling_mean(window_size=ROLL_WINDOW, min_samples=1)
        .over("Final_Account").alias(f"{col}_trailing_{ROLL_WINDOW}w_mean")
    )
    roll_std_exprs.append(
        pl.col(col).shift(1).rolling_std(window_size=ROLL_WINDOW, min_samples=1)
        .over("Final_Account").alias(f"{col}_trailing_{ROLL_WINDOW}w_std")
    )

behavioral_features = behavioral_features.with_columns(lag_exprs + roll_mean_exprs + roll_std_exprs)

delta_exprs = []
ratio_exprs = []
zscore_exprs = []

for col in amount_cols:
    baseline_mean = pl.col(f"{col}_trailing_{ROLL_WINDOW}w_mean").fill_null(0)
    baseline_std = pl.col(f"{col}_trailing_{ROLL_WINDOW}w_std").fill_null(0)

    delta_exprs.append(
        (pl.col(col) - baseline_mean).alias(f"{col}_delta")
    )
    ratio_exprs.append(
        (pl.col(col) / (baseline_mean + 1e-6)).alias(f"{col}_ratio")
    )
    zscore_exprs.append(
        pl.when(baseline_std > 0)
        .then((pl.col(col) - baseline_mean) / baseline_std)
        .otherwise(0.0)
        .alias(f"{col}_zscore")
    )

behavioral_features = behavioral_features.with_columns(delta_exprs + ratio_exprs + zscore_exprs)

output_path_final = "/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/Behavioral_Features_Account_Level_Weekly_WithDeltas_TEST.parquet"
behavioral_features.write_parquet(output_path_final, compression="zstd")
print(f"✅ Weekly features with lag/rolling/delta/ratio/z-score saved to:\n{output_path_final}")
print(f"Total rows: {behavioral_features.height}")
print(f"Total columns: {behavioral_features.width}")

✅ Weekly features with lag/rolling/delta/ratio/z-score saved to:
/content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/Behavioral_Features_Account_Level_Weekly_WithDeltas_TEST.parquet
Total rows: 29391544
Total columns: 51


In [ ]:
print("Columns:", behavioral_features.columns)

print("\nNulls per column:")
print(behavioral_features.null_count())

print("\nAccounts with zero activity across all weeks:")
print(behavioral_features.filter(
    (pl.col("outgoing_transaction_count") == 0) & (pl.col("incoming_transaction_count") == 0)
).height)

print("\nSample of one account's weekly trend (max_amount_sent):")
sample_acct = behavioral_features.select("Final_Account").row(0)[0]
print(
    behavioral_features
    .filter(pl.col("Final_Account") == sample_acct)
    .select([
        "Week", "max_amount_sent", "max_amount_sent_prev_week",
        "max_amount_sent_trailing_4w_mean", "max_amount_sent_delta",
        "max_amount_sent_ratio", "max_amount_sent_zscore"
    ])
)

print("\nTop 10 spikes by zscore (potential anomalies):")
print(
    behavioral_features
    .sort("max_amount_sent_zscore", descending=True)
    .select(["Final_Account", "Week", "max_amount_sent", "max_amount_sent_zscore"])
    .head(10)
)

Columns: ['Final_Account', 'Week', 'outgoing_transaction_count', 'total_amount_sent', 'avg_amount_sent', 'max_amount_sent', 'unique_receivers', 'outgoing_transaction_frequency', 'Week_right', 'incoming_transaction_count', 'total_amount_received', 'avg_amount_received', 'max_amount_received', 'unique_senders', 'incoming_transaction_frequency', 'max_amount_sent_prev_week', 'max_amount_received_prev_week', 'total_amount_sent_prev_week', 'total_amount_received_prev_week', 'avg_amount_sent_prev_week', 'avg_amount_received_prev_week', 'max_amount_sent_trailing_4w_mean', 'max_amount_received_trailing_4w_mean', 'total_amount_sent_trailing_4w_mean', 'total_amount_received_trailing_4w_mean', 'avg_amount_sent_trailing_4w_mean', 'avg_amount_received_trailing_4w_mean', 'max_amount_sent_trailing_4w_std', 'max_amount_received_trailing_4w_std', 'total_amount_sent_trailing_4w_std', 'total_amount_received_trailing_4w_std', 'avg_amount_sent_trailing_4w_std', 'avg_amount_received_trailing_4w_std', 'max_am